In [2]:
import torch
import math
import time
import functools
import contextlib
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

In [ ]:
# 确定是否有GPU环境
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("警告：未检测到GPU，Flashattention需要GPU才可以进行加速。")

GPU Device: Tesla T4
CUDA Version: 12.8


## 注意力机制


`sdpa_kernel()` 是Pytorch用来控制attention后端选择的上下文管理器，支持的后端选择有：

 - SDPBackend.MATH (标准的Attention实现)
 - SDPBackend.FLASH_ATTENTION (FlashAttention实现)
 - SDPBackend.EFFICIENT_ATTENTION (Memory-efficient Attention实现)

 >sdpa_kernel()只是偏好指定，不是强制执行，如果执行FlashAttention的条件不满足的话也会fallback到标准Attention的。


In [ ]:
# 标准的Attention
def attention(q, k, v, is_causal=True):
  with sdpa_kernel(SDPBackend.MATH):
    return F.scaled_dot_product_attention(q, k, v, is_causal=is_causal)


# FlashAttention实现
def flash_attention(q, k, v, is_causal=True):
  with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
    return F.scaled_dot_product_attention(q, k, v, is_causal=is_causal)

## 验证标准Attention和flashAttention的计算结果



In [ ]:
# B->批次大小
# H->注意力头的数量
# L->序列长度，处理文本中的token总数量
# D->每个注意力头的特征维度

B, H, L, D = 16, 32, 512, 128

# q、k、v是计算Attention使用的3个基本元素，代表意义并不一样，数值也基本不一样
q = torch.randn(B, H, L, D)
k = torch.randn(B, H, L, D)
v = torch.randn(B, H, L, D)

# 执行不同的Attention
out_original = attention(q, k, v)
out_flash = flash_attention(q, k, v)

# 计算最大差异diff
diff = (out_original - out_flash).abs().max().item()
print("最大差异值：", diff)

最大差异值： 2.5033950805664062e-06


## 测试

在GPU上对某种Attention执行repeats次，测量平均执行时间

In [ ]:
def benchmark_attention(mode, q, k, v, warmup=1, repeats=3):
    # 1. 预热：运行该函数10次且不计时，以此稳定显卡状态。
    for _ in range(warmup):
        if mode == 'naive':
            attention(q, k, v)
        elif mode == 'flash':
            flash_attention(q, k, v)

    torch.cuda.synchronize() #如果不加同步，你后续进行计时或内存统计时，结果可能会不准确。

    # 2. 记时
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record() # 从这里开始记时
    for _ in range(repeats):
        if mode == 'naive':
            attention(q, k, v)
        elif mode == 'flash':
            flash_attention(q, k, v)
    end_event.record() # 到这里结束记时

    torch.cuda.synchronize() # 等整个重复次数的Attention完成repeats次后，再读取时间。

    # 计算平均延迟（毫秒）
    avg_latency_ms = start_event.elapsed_time(end_event) / repeats

    return avg_latency_ms

## 使用不同的Seq测试对比

In [ ]:
SEQ_LENS = [64, 128, 256, 512, 1024, 2048, 4096]
B, H, D = 4, 8, 64

print(f"Starting Benchmarks ... Params: B={B}, H={H}, D={D}")
print("-" * 90)
print(f"{'Seq Len':<10} | {'Naive (ms)':<12} | {'Flash (ms)':<12} | {'Flash Speedup':<15}")
print("-" * 90)

for L in SEQ_LENS:
    # Generate random input tensors
    q = torch.randn(B, H, L, D)
    k = torch.randn(B, H, L, D)
    v = torch.randn(B, H, L, D)

    # Naive Attention
    lat_naive = benchmark_attention('naive', q, k, v)
    math_str = f"{lat_naive:.2f}"

    torch.cuda.empty_cache()

    # Flash Attention
    lat_flash = benchmark_attention('flash', q, k, v)
    flash_str = f"{lat_flash:.2f}"

    torch.cuda.empty_cache()

    # Calculate Speedup (Naive / Flash)
    speedup = "-"
    s = lat_naive / lat_flash
    speedup = f"{s:.2f}x"

    # Print progress
    print(f"{L:<10} | {math_str:<12} | {flash_str:<12} | {speedup:<15}")

print("-" * 90)
print("Benchmark completed.")

Starting Benchmarks ... Params: B=4, H=8, D=64
------------------------------------------------------------------------------------------
Seq Len    | Naive (ms)   | Flash (ms)   | Flash Speedup  
------------------------------------------------------------------------------------------
64         | 2.37         | 0.66         | 3.58x          
128        | 5.54         | 1.80         | 3.07x          
256        | 21.79        | 6.12         | 3.56x          
512        | 90.43        | 22.97        | 3.94x          
1024       | 399.85       | 101.84       | 3.93x          
2048       | 1601.88      | 233.38       | 6.86x          
4096       | 7609.78      | 933.10       | 8.16x          
------------------------------------------------------------------------------------------
Benchmark completed.


## 使用真实的语言模型来测试Attention

&emsp;&emsp;对于参数规模较小的语言模型，在常见的小 batch、短序列等设置下，其性能瓶颈往往不在显存带宽（HBM访问），而更可能受到计算开销、kernel启动开销、小任务调度（ $launch-bound$ ）或并行利用率不足的影响，因此整体并不处于典型的 $memory-bound$ 状态。

&emsp;&emsp;相比之下，大规模语言模型（尤其在长序列和大batch下）更容易受到显存带宽限制（ $memory-bound$ ），此时 attention计算往往成为 $memory-bound$ 操作。

>**FlashAttention的核心优化**——在于通过重排计算流程、使用tiling和online softmax等方法减少对HBM的读写次数、中间Tensor读写（计算过程中的Attention weight），以“增加一定计算复杂度（IO）”为代价换取更低的内存访问开销，从而缓解 $memory-bound$ 。

&emsp;&emsp;因此，在已经处于 $memory-bound$ 的大模型场景中，FlashAttention通常可以显著提升性能；而在尚未达到带宽瓶颈的小模型场景中，由于其额外的计算与调度开销，反而可能不如标准Attention高效。

>一个总结来看模型的计算效率可以由3个因素共同决定：
$$\max(compute-time, launch-bound, memory-bound)
$$
>
> *   小参数模型：`compute-time && launch-bound > memory-bound`
>
> *   大参数模型：`memory-bound >> compute-time && launch-bound`




## 使用真实的语言模型来测试Attention

&emsp;&emsp;对于参数规模较小的语言模型，在常见的小 batch、短序列等设置下，其性能瓶颈往往不在显存带宽（HBM访问），而更可能受到计算开销、kernel启动开销或并行利用率不足的影响，因此整体并不处于典型的memory-bound状态。

&emsp;&emsp;相比之下，大规模语言模型（尤其在长序列和大batch下）更容易受到显存带宽限制，此时 attention计算往往成为memory-bound操作。

>**FlashAttention的核心优化**——在于通过重排计算流程、使用tiling和online softmax等方法减少对HBM的读写次数、中间Tensor读写（计算过程中的Attention weight），以“增加一定计算复杂度（IO）”为代价换取更低的内存访问开销，从而缓解 $memory_{bound}$ 。

&emsp;&emsp;因此，在已经处于memory-bound的大模型场景中，FlashAttention通常可以显著提升性能；而在尚未达到带宽瓶颈的小模型场景中，由于其额外的计算与调度开销，反而可能不如标准Attention高效。

>一个总结来看模型的计算效率可以由3个因素共同决定：
$$\max(compute_{time}, launch_{bound}, memory_{bound})
$$



In [ ]:
!pip install --upgrade transformers

In [ ]:
from transformers import pipeline

model_id = "opendatalab/MinerU2.5-Pro-2604-1.2B"
print(f"使用真实的LLM: {model_id} ...")

print("下载模型中...")

# 表示使用标准Attention的LLM
pipe_eager = pipeline(
  "text-generation",
  model=model_id,
  model_kwargs={"attn_implementation": "eager"}
)


# 表示使用FlashAttention的LLM
pipe_sdpa = pipeline(
  "text-generation",
  model=model_id,
  model_kwargs={"attn_implementation": "sdpa"}
)

In [ ]:
long_text = "你好，世界！" * 1000 # 改变数值试试看
print(f"输入的数值长度: {len(long_text)} 字元\n")

输入的数值长度: 6000 字元



In [55]:
print(f"=== 测试 Attention Implementation: 标准Attention ===")

print("正在进行 Warmup...")
pipe_eager("Hello", max_new_tokens=1)

print("开始测试...")
start_time = time.time()

# 计算生成一个token消耗的时间（输出延迟）
pipe_eager(long_text,max_new_tokens=1)

# GPU是异步的，python程序执行完成≠GPU正真执行完成
torch.cuda.synchronize()

# 等GPU真正执行完成才结束记时
end_time = time.time()
latency = end_time - start_time
print(f"耗时: {latency:.4f} 秒")

# 清理记忆体
torch.cuda.empty_cache()

Both `max_new_tokens` (=1) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== 测试 Attention Implementation: 标准Attention ===
正在进行 Warmup...
开始测试...
耗时: 2.5046 秒


In [56]:
print(f"=== 测试 Attention Implementation: FlashAttention ===")

print("正在进行 Warmup...")
pipe_sdpa("Hello", max_new_tokens=1)

print("开始测试...")
start_time = time.time()

pipe_sdpa(long_text,max_new_tokens=1)

torch.cuda.synchronize()

end_time = time.time()
latency = end_time - start_time
print(f"耗时: {latency:.4f} 秒")

# 清理记忆体
torch.cuda.empty_cache()

Both `max_new_tokens` (=1) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== 测试 Attention Implementation: FlashAttention ===
正在进行 Warmup...
开始测试...
耗时: 2.5287 秒
